# 04 — Model Evaluation

Full evaluation suite for the ChainScore credit risk models.

**Metrics (industry standard in credit risk):**
| Metric | Description | Typical benchmark |
|---|---|---|
| ROC-AUC | Overall discrimination ability | > 0.70 good, > 0.80 very good |
| KS Statistic | Max separation between class CDFs | > 0.30 good for credit risk |
| Gini Coefficient | 2*AUC - 1, preferred by some banks | > 0.40 good |
| Brier Score | Calibration quality (lower is better) | Context-dependent |
| Lift at D1 | How much better than random at top 10% | > 2x is useful |

**Plots generated:**
- ROC curves with AUC
- Calibration reliability diagram  
- KS plot (class separation)
- Lift chart by decile
- SHAP feature importance (beeswarm)
- Score distribution by class

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.metrics import roc_auc_score, brier_score_loss
from src.models.evaluate import (
    ks_statistic, gini_coefficient, lift_at_decile,
    plot_roc_curves, plot_calibration, plot_ks,
    plot_lift, plot_shap_summary, plot_score_distribution,
)
from src.models.train import load_and_split

sns.set_theme(style='whitegrid', context='notebook')
BRAND_BLUE = '#185FA5'
REPORTS_DIR = Path('../reports/figures')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load models and test data

In [ ]:
MODELS_DIR = Path('../models')
FEATURE_PATH = Path('../data/processed/feature_matrix.parquet')

with (MODELS_DIR / 'logistic_regression.pkl').open('rb') as f:
    lr = pickle.load(f)
with (MODELS_DIR / 'lightgbm.pkl').open('rb') as f:
    lgb_model = pickle.load(f)
with (MODELS_DIR / 'feature_columns.json').open() as f:
    feature_cols = json.load(f)

X_train, X_test, y_train, y_test, _ = load_and_split(FEATURE_PATH)
y_test_np = y_test.values

lr_prob  = lr.predict_proba(X_test)[:, 1]
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]

print(f'Test set: {len(X_test):,} wallets | Default rate: {y_test_np.mean():.2%}')

## 2. Summary metrics table

In [ ]:
summary = []
for name, prob in [('Logistic Regression', lr_prob), ('LightGBM', lgb_prob)]:
    summary.append({
        'Model':    name,
        'ROC-AUC':  round(roc_auc_score(y_test_np, prob), 4),
        'KS':       round(ks_statistic(y_test_np, prob), 4),
        'Gini':     round(gini_coefficient(y_test_np, prob), 4),
        'Brier':    round(brier_score_loss(y_test_np, prob), 4),
        'Lift@D1':  round(lift_at_decile(y_test_np, prob, 1), 2),
        'Lift@D2':  round(lift_at_decile(y_test_np, prob, 2), 2),
    })

summary_df = pd.DataFrame(summary).set_index('Model')
print('\nModel Performance Summary:')
print('=' * 60)
print(summary_df.to_string())

## 3. ROC curves

In [ ]:
probs = {'Logistic Regression': lr_prob, 'LightGBM': lgb_prob}
plot_roc_curves(y_test_np, probs, REPORTS_DIR / 'roc_curves.png')
plt.show()

## 4. Calibration plot

A well-calibrated model should have its reliability curve close to the diagonal.
If predicted probability = 0.3, approximately 30% of those wallets should actually default.
This is critical for pricing — an uncalibrated PD leads to mispriced loans.

In [ ]:
plot_calibration(y_test_np, probs, REPORTS_DIR / 'calibration_plot.png')
plt.show()

## 5. KS plot (LightGBM)

The KS statistic is the most widely cited metric in consumer credit scoring in Brazil and LatAm.
It measures the maximum vertical distance between the cumulative distributions of
default and non-default wallets.

In [ ]:
plot_ks(y_test_np, lgb_prob, 'LightGBM', REPORTS_DIR / 'ks_plot.png')
plt.show()

## 6. Lift chart

Shows how many times more defaults are captured in the top N% of predicted risk
compared to random selection. A Lift@D1 of 3x means the top 10% of wallets ranked
by our model contains 3x more actual defaults than a random 10%.

In [ ]:
plot_lift(y_test_np, probs, REPORTS_DIR / 'lift_chart.png')
plt.show()

## 7. SHAP feature importance (LightGBM)

SHAP (SHapley Additive exPlanations) assigns each feature a contribution to each
individual prediction. This is the gold standard for model explainability and is
increasingly required by regulators for credit decisions.

In [ ]:
plot_shap_summary(lgb_model, X_test, REPORTS_DIR / 'shap_summary.png')
plt.show()

## 8. Score distribution

Shows how ChainScore values (0–1000) are distributed for default vs non-default wallets.
Good separation between classes means the score is useful for risk stratification.

In [ ]:
plot_score_distribution(y_test_np, lgb_prob, REPORTS_DIR / 'score_distribution.png')
plt.show()

## 9. Scorecard interpretation (Logistic Regression)

The logistic regression coefficients represent log-odds contributions per unit change
in each feature. This is the same logic used in traditional FICO-style scorecards.

In [ ]:
# Extract LR coefficients from the calibrated pipeline
try:
    inner_lr = lr.named_steps['classifier'].calibrated_classifiers_[0].estimator
    coefs = pd.Series(inner_lr.coef_[0], index=feature_cols)
    top_pos = coefs.nlargest(10)
    top_neg = coefs.nsmallest(10)
    top_coefs = pd.concat([top_pos, top_neg]).sort_values()

    fig, ax = plt.subplots(figsize=(9, 7))
    colors = ['#E05C2A' if v > 0 else BRAND_BLUE for v in top_coefs.values]
    ax.barh(top_coefs.index, top_coefs.values, color=colors)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title('Logistic Regression — Top 20 Feature Coefficients')
    ax.set_xlabel('Log-odds coefficient (positive = increases default probability)')
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'lr_coefficients.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Could not extract coefficients: {e}')

## 10. Save full evaluation results

In [ ]:
eval_results = {}
for name, prob in [('logistic_regression', lr_prob), ('lightgbm', lgb_prob)]:
    eval_results[name] = {
        'roc_auc':   float(roc_auc_score(y_test_np, prob)),
        'ks':        float(ks_statistic(y_test_np, prob)),
        'gini':      float(gini_coefficient(y_test_np, prob)),
        'brier':     float(brier_score_loss(y_test_np, prob)),
        'lift_d1':   float(lift_at_decile(y_test_np, prob, 1)),
        'lift_d2':   float(lift_at_decile(y_test_np, prob, 2)),
    }

with (REPORTS_DIR / 'evaluation_results.json').open('w') as f:
    json.dump(eval_results, f, indent=2)

print('Evaluation complete. All results saved to reports/figures/')